# Getting Started with OPERA DSWx-HLS Products
## Streaming and visualizing Cloud-Optimized Geotiff (COG) OPERA DSWx-HLS products using CMR's SpatioTemporal Asset Catalog (CMR-STAC)
### This tutorial demonstrates how to query and work with the OPERA DSWx-HLS products from the cloud ([OPERA_L3_DSWX-HLS_V1](https://dx.doi.org/10.5067/OPDSW-PL3V0)).

---    

### Data Used in the Example  

- **30 meter (m) global OPERA Dynamic Surface Water Extent from Harmonized Landsat Sentinel-2A/B product (Version 1) - [OPERA_L3_DSWX-HLS_V1](https://dx.doi.org/10.5067/OPDSW-PL3V0)**
    - This dataset contains OPERA Level-3 Dynamic Surface Water Extent product version 1. The input dataset for generating each product is the Harmonized Landsat-8 and Sentinel-2A/B (HLS) product version 2.0. HLS products provide surface reflectance (SR) data from the Operational Land Imager (OLI) aboard the Landsat 8 satellite and the MultiSpectral Instrument (MSI) aboard the Sentinel-2A/B satellite. The surface water extent products are distributed over projected map coordinates using the Universal Transverse Mercator (UTM) projection. Each UTM tile covers an area of 109.8 km × 109.8 km. This area is divided into 3,660 rows and 3,660 columns at 30-m pixel spacing. Each product is distributed as a set of 10 GeoTIFF (Geographic Tagged Image File Format) files including water classification, associated confidence, land cover classification, terrain shadow layer, cloud/cloud-shadow classification, Digital elevation model (DEM), and Diagnostic layer.
     - **Science Dataset (SDS) layers:**  
        - B01_WTR (Water Classification Layer)  
        - B02_BWTR (Binary Water Layer)  
        - B03_CONF (Confidence Layer)  

Please refer to the [OPERA Product Specification Document](https://d2pn8kiwq2w21t.cloudfront.net/documents/ProductSpec_DSWX_URS309746.pdf) for details about the DSWx-HLS product.

---
## Topics Covered  

1. [**Getting Started**](#getstarted)   
2. [**CMR-STAC API: Search for data based on spatial query**](#searchstac)      
3. [**Load and visualize DSWX-HLS COGs from the Cloud**](#loadandvizdswx)        

---

## Before Starting this Tutorial  

A [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account at the link provided.

---
## 1. Getting Started <a id="getstarted"></a>

### 1.1 Import Libraries
Notebook dependencies may be installed into a self-contained python environment using the `environment.yml` file available in the [OPERA Applications Github repository](https://github.com/OPERA-Cal-Val/OPERA_Applications) or they may be installed manually.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

import os
import earthaccess
from netrc import netrc
from subprocess import Popen
from platform import system
from getpass import getpass

from pystac_client import Client  
from pystac_client import ItemSearch
from pystac.item import Item
from typing import Dict, Any
import json

import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from datetime import datetime
from tqdm import tqdm

from shapely.geometry import box
from shapely.geometry import shape
from shapely.ops import transform

import numpy as np
import pandas as pd
import geopandas as gpd
from skimage import io

from osgeo import gdal
import rasterio as rio

import pyproj
from pyproj import Proj

import xarray as xr
import panel as pn
import geoviews as gv
import hvplot.xarray
import holoviews as hv

from bokeh.models import FixedTicker
hv.extension('bokeh')
gv.extension('bokeh', 'matplotlib')

### 1.2 Authentication with NASA Earthdata credentials
A [NASA Earthdata Login](https://urs.earthdata.nasa.gov/) account is required to download the data used in this tutorial. You can create an account at the link provided. After establishing an account, the code in the next cell will verify authentication. If this is your first time running the notebook, you will be prompted to enter your Earthdata login credentials, which will be saved in ~/.netrc.

In [ ]:
urs = 'urs.earthdata.nasa.gov'    # Earthdata URL endpoint for authentication
prompts = ['Enter NASA Earthdata Login Username: ',
           'Enter NASA Earthdata Login Password: ']

# Determine the OS (Windows machines usually use an '_netrc' file)
netrc_name = "_netrc" if system()=="Windows" else ".netrc"

# Determine if netrc file exists, and if so, if it includes NASA Earthdata Login Credentials
try:
    netrcDir = os.path.expanduser(f"~/{netrc_name}")
    netrc(netrcDir).authenticators(urs)[0]

# Below, create a netrc file and prompt user for NASA Earthdata Login Username and Password
except FileNotFoundError:
    homeDir = os.path.expanduser("~")
    Popen('touch {0}{2} | echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)
    # Set restrictive permissions
    Popen('chmod 0600 {0}{1}'.format(homeDir + os.sep, netrc_name), shell=True)

# Determine OS and edit netrc file if it exists but is not set up for NASA Earthdata Login
except TypeError:
    homeDir = os.path.expanduser("~")
    Popen('echo machine {1} >> {0}{2}'.format(homeDir + os.sep, urs, netrc_name), shell=True)
    Popen('echo login {} >> {}{}'.format(getpass(prompt=prompts[0]), homeDir + os.sep, netrc_name), shell=True)
    Popen('echo \'password {} \'>> {}{}'.format(getpass(prompt=prompts[1]), homeDir + os.sep, netrc_name), shell=True)

The next cell configures the `gdal` library and provideds necessary authentication to successfully access cloud-hosted assets.

In [ ]:
# Set GDAL configs to successfully access Cloud Assets via vsicurl
gdal.SetConfigOption("GDAL_HTTP_UNSAFESSL", "YES")
gdal.SetConfigOption('GDAL_HTTP_COOKIEFILE','~/cookies.txt')
gdal.SetConfigOption('GDAL_HTTP_COOKIEJAR', '~/cookies.txt')
gdal.SetConfigOption('GDAL_DISABLE_READDIR_ON_OPEN','FALSE')
gdal.SetConfigOption('CPL_VSIL_CURL_ALLOWED_EXTENSIONS','TIF')

### 1.3 Set up Working Environment <a id="1.2"></a>

In [ ]:
inDir = os.getcwd()
os.chdir(inDir)

## 2. CMR-STAC API: Search for data based on spatial query <a id="searchstac"></a>

### 2.1 Initialize user-defined parameters <a id="2.1"></a>

In [ ]:
aoi = box(67.4, 26.2, 68.0, 26.8)
start_date = datetime(2025, 1, 1)
stop_date = datetime(2025, 7, 31)        
overlap_threshold = 50                                                  # in percent
cloud_cover_threshold = 40                                             # in percent

print(f"Search between {start_date} and {stop_date}")
print(f"With AOI: {aoi.__geo_interface__}")

In [ ]:
# Search data through CMR-STAC API
stac = 'https://cmr.earthdata.nasa.gov/cloudstac'    # CMR-STAC API Endpoint
api = Client.open(f'{stac}/POCLOUD/')
collections = ['OPERA_L3_DSWX-S1_V1_1.0']

search_params = {"collections": collections,
                 "intersects": aoi.__geo_interface__,
                 "datetime": [start_date, stop_date],
                 "limit": 50,
                 "max_items": 100
                 }
search_dswx = api.search(**search_params)

### 2.2 Query DSWx-HLS tiles based on spatial overlap with respect to defined AOI <a id="2.2"></a>

In [ ]:
# Function to calculate percentage overlap between user-defined bbox and dswx tile
def intersection_percent(item: Item, aoi: Dict[str, Any]) -> float:
    '''The percentage that the Item's geometry intersects the AOI. An Item that
    completely covers the AOI has a value of 100.
    '''
    geom_item = shape(item.geometry)
    geom_aoi = shape(aoi)
    intersected_geom = geom_aoi.intersection(geom_item)
    intersection_percent = (intersected_geom.area * 100) / geom_aoi.area

    return intersection_percent

In [ ]:
# Filter datasets based on spatial overlap 
intersects_geometry = aoi.__geo_interface__

#Check percent overlap values
print("Percent overlap before filtering: ")
print([f"{intersection_percent(i, intersects_geometry):.2f}" for i in search_dswx.items()])

# Apply spatial overlap
dswx_filtered = (
    i for i in search_dswx.items() if intersection_percent(i, intersects_geometry) > overlap_threshold 
)

In [ ]:
# Inspect the items inside the filtered query
dswx_data = list(dswx_filtered)
# Inspect one data
dswx_data[0].to_dict()

In [ ]:
## Print search information
# Tota granules
print(f"Total granules after search filter: {len(dswx_data)}")

#Check percent overlap values
print("Percent-overlap: ")
print([f"{intersection_percent(i, intersects_geometry):.2f}" for i in dswx_data])

In [ ]:
# Visualize the DSWx tile boundary and the user-defined bbox
geom_df = []
for d,_ in enumerate(dswx_data):
    geom_df.append(shape(dswx_data[d].geometry))

geom_granules = gpd.GeoDataFrame({'geometry':geom_df})
granules_poly = gv.Polygons(geom_granules, label='DSWx tile boundary').opts(line_color='blue', color=None, show_legend=True)

# Use geoviews to combine a basemap with the shapely polygon of our Region of Interest (ROI)
base = gv.tile_sources.EsriImagery.opts(width=1000, height=1000)

# Get the user-specified aoi
geom_aoi = shape(intersects_geometry)
aoi_poly = gv.Polygons(geom_aoi, label='User-specified bbox').opts(line_color='yellow', color=None, show_legend=True)

# Plot using geoviews wrapper
granules_poly*base*aoi_poly

In [ ]:
# Create table of search results
dswx_data_df = []
for item in dswx_data:
    item.to_dict()
    fn = item.id.split('_')
    ID = fn[3]
    sensor = fn[6]
    dat = item.datetime.strftime('%Y-%m-%d')
    spatial_coverage = intersection_percent(item, intersects_geometry)
    geom = item.geometry

    # Take all the band href information 
    band_links = [item.assets[links].href for links in item.assets.keys()]
    dswx_data_df.append([ID,sensor,dat,geom,spatial_coverage,band_links])

dswx_data_df = pd.DataFrame(dswx_data_df, columns = ['TileID', 'Sensor', 'Date', 'Footprint','SpatialCoverage','BandLinks'])
dswx_data_df

## 3. Load and visualize DSWX-HLS COGs from the Cloud <a id="loadandvizdswx"></a>

In [ ]:
# Check out layers inside one of the datasets
viz_dswx = dswx_data_df.iloc[5]
viz_dswx.BandLinks

### 3.1 Extract DSWx-HLS COGs and subset by Band <a id="3.1"></a>

In [ ]:
# Function to read each layers and stack them to create a geocube
def stack_bands(bandpath:str, bandlist:list): 
    '''
    Returns geocube with three bands stacked into one multi-dimensional array.
            Parameters:
                    bandpath (str): Path to bands that should be stacked
                    bandlist (list): Three bands that should be stacked
            Returns:
                    bandStack (xarray Dataset): Geocube with stacked bands
                    crs (int): Coordinate Reference System corresponding to bands
    '''
    bandStack = []; bandS = []; bandStack_ = []
    for i,band in enumerate(bandlist):
        print(f"Streaming: {bandpath%band}")
        if i==0:
            bandStack_ = xr.open_rasterio(bandpath%band)
            crs = pyproj.CRS.to_epsg(pyproj.CRS.from_proj4(bandStack_.crs))
            bandStack_ = bandStack_ * bandStack_.scales[0]
            bandStack = bandStack_.squeeze(drop=True)
            bandStack = bandStack.to_dataset(name='z')
            bandStack.coords['band'] = i+1
            bandStack = bandStack.rename({'x':'longitude', 'y':'latitude', 'band':'band'})
            bandStack = bandStack.expand_dims(dim='band')  
        else:
            bandS = xr.open_rasterio(bandpath%band)
            bandS = bandS * bandS.scales[0]
            bandS = bandS.squeeze(drop=True)
            bandS = bandS.to_dataset(name='z')
            bandS.coords['band'] = i+1
            bandS = bandS.rename({'x':'longitude', 'y':'latitude', 'band':'band'})
            bandS = bandS.expand_dims(dim='band')
            bandStack = xr.concat([bandStack, bandS], dim='band')
    return bandStack, crs

In [ ]:
# Take the URLs of the bands
data_dir = viz_dswx.BandLinks[3].split('v1.0')[0]
file_ext = 'tif'
bandlist = ['B01_WTR', 'B02_BWTR', 'B03_CONF']
bandpath = f"{data_dir}v1.0_%s.{file_ext}"

### 3.2 Stack the bands <a id="1.5"></a>

In [ ]:
# Creates geocube of stacked bands
da, crs = stack_bands(bandpath, bandlist)

# Creates basemap using geoviews
base = gv.tile_sources.EsriImagery.opts(width=1000, height=1000, padding=0.1)

In [ ]:
# Mask nodata values (255)
da_masked = da.where(da['z'] != 255.) 
B01_WTR = da_masked.z.sel({'band':1})
B02_BWTR = da_masked.z.sel({'band':2}) 
B03_CONF = da_masked.z.sel({'band':3}) 

### 3.3 Visualize the bands on a map <a id="3.2"></a>

In [ ]:
# Visualize B01 - WATER CLASSIFICATION LAYER
# Parameters for Colorbar
levels = [0, 0.9, 1.9, 2.9, 7.9, 8.9, 10]
color_key = {
    "Not Water": "#ffffff",
    "Open Water": "#0000ff",
    "Partial Surface Water": "#00ff00",
    "Reserved": "#000000",
    "Snow/Ice": "#00ffff",
    "Clouds/Cloud Shadow": "#7f7f7f"
}

ticks = [0.5, 1.5, 2.5, 5.5, 8.5, 9.5]
ticker = FixedTicker(ticks=ticks)
labels = dict(zip(ticks, color_key))

fig = B01_WTR.hvplot.image(x='longitude', 
                          y='latitude', 
                          crs=crs, 
                          rasterize=False,      #True if running in binder for faster calculation. By default, should be False to deactivate interpolation.
                          dynamic=False,        #True if running in binder for faster calculation.
                          aspect='equal', 
                          frame_width=500,
                          clim=(0,255),
                          frame_height=500, 
                          alpha=0.6).opts(title=f"B01 WTR", xlabel='Longitude', ylabel='Latitude', color_levels= levels, cmap=tuple(color_key.values()), 
                                        colorbar_opts={'ticker': ticker, 'major_label_overrides': labels}, clim=(0,10)) * base
#Uncomment to save
#hvplot.save(fig, 'B01_WTR.png')
fig

In [ ]:
# Visualize B02 - BINARY WATER LAYER
# Parameters for Colorbar
levels = [0, 0.9, 1.9, 7.9, 8.9, 10]
color_key = {
    "Not Water": "#ffffff",
    "Water": "#0000ff",
    "Reserved": "#000000",
    "Snow/Ice": "#00ffff",
    "Clouds/Cloud Shadow": "#7f7f7f"
}

ticks = [0.5, 1.5, 5.5, 8.5, 9.5]
ticker = FixedTicker(ticks=ticks)
labels = dict(zip(ticks, color_key))

fig = B02_BWTR.hvplot.image(x='longitude', 
                          y='latitude', 
                          crs=crs, 
                          rasterize=False,      #True if running in binder for faster calculation. By default, should be False to deactivate interpolation. 
                          dynamic=False,        #True if running in binder for faster calculation.
                          aspect='equal', 
                          frame_width=500,
                          clim=(0,255),
                          frame_height=500, 
                          alpha=0.6).opts(title=f"B02 BWTR", xlabel='Longitude', ylabel='Latitude', color_levels= levels, cmap=tuple(color_key.values()), 
                                        colorbar_opts={'ticker': ticker, 'major_label_overrides': labels}, clim=(0,10)) * base
#Uncomment to save
#hvplot.save(fig, 'B02_BWTR.png')
fig

In [ ]:
# Visualize B03 - CONFIDENCE LAYER
# Parameters for Colorbar
with rio.open(f"{data_dir}v1.0_B03_CONF.tif") as ds:
    colormap = ds.colormap(1)

color_key = ListedColormap([np.array(colormap[key]) / 255 for key in range(256)])
ticks = [0, 50, 100, 175, 245, 255]
ticker = FixedTicker(ticks=ticks)
labels = dict(zip(ticks, ["0", "Confidence", "100", "Reserved", "Snow/Ice", "Clouds/Cloud Shadow"]))

fig = B03_CONF.hvplot.image(x='longitude', 
                          y='latitude', 
                          crs=crs, 
                          rasterize=False,      #True if running in binder for faster calculation. By default, should be False to deactivate interpolation.
                          dynamic=False,        #True if running in binder for faster calculation. 
                          aspect='equal', 
                          frame_width=500,
                          clim=(0,255),
                          frame_height=500, 
                          alpha=0.6).opts(title=f"B03 CONF", xlabel='Longitude', ylabel='Latitude', cmap=color_key,
                                    colorbar_opts={'ticker': ticker, 'major_label_overrides': labels}) * base
#Uncomment to save
#hvplot.save(fig, 'B03_CONF.png')
fig